In [5]:
%idle_timeout 30
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job

print('Bibliotecas importadas')

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: 0ae88ce8-50ce-40ea-b00e-028863f03f19
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 0ae88ce8-50ce-40ea-b00e-028863f03f19 to get into ready status...


In [ ]:
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init("teste-interativo-gold")

DATABASE = "state_of_data_db"
BUCKET = "pos-tech-state-of-data-grupo-72"

print("Consultando dados da camada Silver")

df_silver = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE,
    table_name="silver_state_of_data"
).toDF()

df_silver.createOrReplaceTempView("silver_view")

print('Silver lida e Temp view criada')

In [ ]:
print('Criando camada Gold com Spark.sql')

df_gold = spark.sql("""
SELECT *
FROM state_of_data_db.silver_state_of_data
WHERE (LOWER(codigo_1), LOWER(codigo_2)) IN (
    ('4', 'g'),
    ('4', 'f'),
    ('7', 'b'),
    ('8', 'c'),
    ('4', 'm'),
    ('4', 'j'),
    ('3', 'e'),
    ('2', 'n'),
    ('2', 'l'),
    ('2', 't'),
    ('2', 's'),
    ('2', 'o')
)
""")

# Salvando no S3 e catalogando no Data Catalog

caminho_gold = f"s3://{BUCKET}/gold/fato_analise_pbi/"

# Limpa qualquer dado anterior antes de escrever (evita duplicação por múltiplas execuções)
glueContext.purge_s3_path(caminho_gold, options={"retentionPeriod": 0})

dyf_gold = DynamicFrame.fromDF(df_gold, glueContext, "dyf_gold")

sink = glueContext.getSink(
    connection_type="s3",
    path=caminho_gold,
    enableUpdateCatalog=True
)
sink.setFormat("glueparquet")
sink.setCatalogInfo(catalogDatabase=DATABASE, catalogTableName="gold_fato_analise_pbi")
sink.writeFrame(dyf_gold)

print("Gold gravada e catalogada com sucesso.")
job.commit()